# Learning over Text

Brandeis University COSI 104A Spring 26 Xinyi Fang & Professor Dylan Cashman

In this notebook, we will learn how to use BoW and TF-IDF as features of Linear Regression.

We will be using the [Fake News Detection Dataset](https://www.kaggle.com/datasets/bhavikjikadara/fake-news-detection?select=true.csv) from Kaggle, for which we are going to tackle the task of classifying news as real or fake.

NOTE - you have to download the files `true.csv` and `false.csv` from Kaggle to run this code.  We do not include them in the repository because they are both about 50 mb.

### Load and process the dataset

Let's check the dataset first. After downloading the dataset, we load it and using nltk library to preprocess the data. Then we can make wordclouds according to the frequency distribution of the words, but you can also try what would happen if we are not removing the stop words etc.

In [ ]:
# !pip install nltk
# !pip install wordcloud
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import nltk
from wordcloud import WordCloud

# we will be using these necessary nltk resources
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# load the datasets
true_news_df = pd.read_csv('true.csv')
fake_news_df = pd.read_csv('fake.csv')

true_news_df.head()

As we can notice, it is not a good dataset since every real news text begins with (Reuters). But we can still try to distinguish the news title.

In [ ]:
fake_news_df.head()

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    word_tokens = word_tokenize(text)
    # Lemmatize and remove stop words and non-alphabetic characters
    lemmatized_output = [lemmatizer.lemmatize(w) for w in word_tokens if not w.lower() in stop_words and w.isalpha()]
    return lemmatized_output

# preprocess the titles
true_news_df['processed'] = true_news_df['title'].apply(preprocess_text)
fake_news_df['processed'] = fake_news_df['title'].apply(preprocess_text)

true_news_df['processed'].head()

In [ ]:
# concatenate all the processed words
all_true_words = sum(true_news_df['processed'].tolist(), [])
all_fake_words = sum(fake_news_df['processed'].tolist(), [])

# get the frequency distribution of the words
true_word_freq = nltk.FreqDist(all_true_words)
fake_word_freq = nltk.FreqDist(all_fake_words)

true_word_freq

### Make word clouds
We can use the frequency distribution of the words to make word clouds so we can intuitively identify the commonly used words.

In [ ]:
import matplotlib.pyplot as plt

# get the top 100 words
top100_true_words = true_word_freq.most_common(100)
top100_fake_words = fake_word_freq.most_common(100)

# create DataFrame from the top 100 words
top_words_df = pd.DataFrame({
    'Real_News_Word': [word for word, freq in top100_true_words],
    'Real_News_Frequency': [freq for word, freq in top100_true_words],
    'Fake_News_Word': [word for word, freq in top100_fake_words],
    'Fake_News_Frequency': [freq for word, freq in top100_fake_words],
})

# save to csv
top_words_df.to_csv('TOP100title.csv', index=False)

# generate word cloud images
def generate_word_cloud(frequencies, filename):
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(dict(frequencies))
    wordcloud.to_file(filename + '.png')
    return wordcloud

# generate word clouds for both real and fake news
wc_real = generate_word_cloud(top100_true_words, 'real_news_wordcloud')
wc_fake = generate_word_cloud(top100_fake_words, 'fake_news_wordcloud')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(wc_real, interpolation='bilinear')
axes[0].set_title('Real News Word Cloud', fontsize=16)
axes[0].axis('off')

axes[1].imshow(wc_fake, interpolation='bilinear')
axes[1].set_title('Fake News Word Cloud', fontsize=16)
axes[1].axis('off')

plt.tight_layout()
plt.show()

### BoW and TF-IDF Implementation
Then we want to try out BoW and TF-IDF we learned today. We need to determine the label first. Here, we set true news as 1, fake news as 0.
We also need to split the data ourselves. 

In [ ]:
# add labels: 1 for True News, 0 for Fake News
true_news_df['label'] = 1
fake_news_df['label'] = 0

# combine the datasets
df = pd.concat([true_news_df, fake_news_df], ignore_index=True)

# scikit-Learn vectorizers expect strings, not lists of words. 
# so we join our lemmatized tokens back into sentences.
df['processed_text'] = df['processed'].apply(lambda x: ' '.join(x))

from sklearn.model_selection import train_test_split

# split data into 70% train and 30% test subsets
X_train, X_test, y_train, y_test = train_test_split(
    df['processed_text'], df['label'], test_size=0.3, random_state=42, shuffle=True
)

print(f"Training data size: {X_train.shape[0]}")
print(f"Testing data size: {X_test.shape[0]}")

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
import matplotlib.pyplot as plt

# initialize vectorizer (limit to 5000 features to keep it fast)
count_vec = CountVectorizer(max_features=5000)

# fit and transform the training Data, transform the test data
X_train_bow = count_vec.fit_transform(X_train)
X_test_bow = count_vec.transform(X_test)

# train the model
clf_bow = LogisticRegression(max_iter=1000)
clf_bow.fit(X_train_bow, y_train)

# predict and evaluate
y_pred_bow = clf_bow.predict(X_test_bow)

print(f"BoW Model Accuracy: {metrics.accuracy_score(y_test, y_pred_bow):.4f}")

# display Confusion Matrix
disp_bow = metrics.ConfusionMatrixDisplay.from_predictions(y_test, y_pred_bow, display_labels=['Fake', 'True'])
disp_bow.figure_.suptitle("Confusion Matrix - Bag of Words")
plt.show()

In [ ]:
import numpy as np

# get the vocabulary (feature names) and the model weights (coefficients)
feature_names_bow = count_vec.get_feature_names_out()
weights_bow = clf_bow.coef_[0]

# create a DataFrame to view words and their corresponding importance
weights_df_bow = pd.DataFrame({
    'Word': feature_names_bow,
    'Weight': weights_bow
})

# words with the highest positive weights push the prediction towards 1 (True News)
top_true_words_bow = weights_df_bow.sort_values(by='Weight', ascending=False).head(10)

# words with the lowest negative weights push the prediction towards 0 (Fake News)
top_fake_words_bow = weights_df_bow.sort_values(by='Weight', ascending=True).head(10)

print("BoW - Top 10 strongest indicators of TRUE news:")
print(top_true_words_bow.to_string(index=False))
print("\n" + "="*40 + "\n")
print("BoW - Top 10 strongest indicators of FAKE news:")
print(top_fake_words_bow.to_string(index=False))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# initialize TF-IDF Vectorizer
tfidf_vec = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf = tfidf_vec.transform(X_test)

clf_tfidf = LogisticRegression(max_iter=1000)
clf_tfidf.fit(X_train_tfidf, y_train)

y_pred_tfidf = clf_tfidf.predict(X_test_tfidf)

print(f"TF-IDF Model Accuracy: {metrics.accuracy_score(y_test, y_pred_tfidf):.4f}")

# display Confusion Matrix
disp_tfidf = metrics.ConfusionMatrixDisplay.from_predictions(y_test, y_pred_tfidf, display_labels=['Fake', 'True'], cmap='Blues')
disp_tfidf.figure_.suptitle("Confusion Matrix - TF-IDF")
plt.show()

In [ ]:
# get the vocabulary (feature names) and the model weights (coefficients)
feature_names = tfidf_vec.get_feature_names_out()
weights = clf_tfidf.coef_[0]

# create a DataFrame to view words and their corresponding importance
weights_df = pd.DataFrame({
    'Word': feature_names,
    'Weight': weights
})

# words with the highest positive weights push the prediction towards 1 (True News)
top_true_words = weights_df.sort_values(by='Weight', ascending=False).head(10)

# words with the lowest negative weights push the prediction towards 0 (Fake News)
top_fake_words = weights_df.sort_values(by='Weight', ascending=True).head(10)

print("TF-IDF - Top 10 strongest indicators of TRUE news:")
print(top_true_words.to_string(index=False))
print("\n" + "="*40 + "\n")
print("TF-IDF - Top 10 strongest indicators of FAKE news:")
print(top_fake_words.to_string(index=False))